# v0.35.0 — Full-text search

## 1. Intro

Since v0.35.0 a queryset searches text with the full SurrealDB-ORM's API, over the full-text indexes a model
declares:

- `Analyzer` is SurrealDB's `DEFINE ANALYZER` — tokenizers and filters — and `Index` its `DEFINE INDEX`: a plain index,
  a `UNIQUE` one, or a full-text one (`search_analyzer=`, `bm25=`, `highlights=`), with the argument names of the full
  ORM's `DefineAnalyzer` and `CreateIndex`;
- a model lists its indexes in `SurrealConfigDict(indexes=[...])`: `index_ddl()` renders their statements without a
  connection, and `await define_indexes()` applies only what is missing or changed, rebuilding the indexes of an
  analyzer it redefines;
- `search(**field_queries)` keeps the records whose fields match — the text to match is bound as a query variable — and
  `SearchScore`, `SearchHighlight` and `SearchOffsets` in `annotate()` give each record its BM25 score, its text with the
  matched terms between tags, and where they sit;
- `remove_index()` removes an index by name, and `SurrealDBConnectionManager.define_analyzer()` / `remove_analyzer()`
  manage the database's analyzers.

The code is the same on SurrealDB 2.x, 3.x and the embedded engine. The index keyword follows the server line:
`FULLTEXT` on SurrealDB 3.x, `SEARCH` on 2.x and the embedded engine, each of which refuses the other's —
`define_indexes()` writes the right one. The notebook runs top to bottom on both lines; the outputs below come from a
SurrealDB 3.x server: a 2.x server prints `SEARCH` where they show `FULLTEXT`, and §8 says where its scores differ.

## 2. Connect

`SURREALDB_HOST` and `SURREALDB_PORT` choose the server (default `localhost:8000`), over WebSocket: the live queries of
§11 need it.

In [1]:
import contextlib
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
# WebSocket (.../rpc): live queries need it.
SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 3. An analyzer and a model

`FTS_ENGLISH` splits the text on case changes and blanks (`class`, `blank`), then lower-cases each token, folds its
accents and stems it (`lowercase`, `ascii`, `snowball(english)`); a search text goes through the same analyzer. An
analyzer belongs to the database: declare one per name, a module constant, and reference it from every index that
uses it.

`Article` declares three indexes: `title`, full-text with BM25 scores and highlights; `body`, full-text with BM25
scores; `slug`, `UNIQUE`. The indexes are checked when the class is defined — distinct names, one full-text index per
field, one `Analyzer` per analyzer name — and `get_indexes()` returns them in declaration order.

The cell first removes what an earlier run left — the tables, with their indexes, then the analyzers — so the
notebook can be re-executed in place. `major` is the server's major version, from `client.version()`.

In [2]:
from surreal_orm_lite import Analyzer, BaseSurrealModel, Index, SurrealConfigDict

FTS_ENGLISH = Analyzer("nb_english", ["class", "blank"], ["lowercase", "ascii", "snowball(english)"])


class Article(BaseSurrealModel):
    model_config = SurrealConfigDict(
        table_name="nb_article",
        indexes=[
            Index("nb_article_title", "title", search_analyzer=FTS_ENGLISH, bm25=True, highlights=True),
            Index("nb_article_body", "body", search_analyzer=FTS_ENGLISH, bm25=True),
            Index("nb_article_slug", "slug", unique=True),
        ],
    )

    id: str | None = None
    title: str
    body: str = ""
    slug: str
    topic: str = ""


RESET = [
    "REMOVE TABLE IF EXISTS nb_article;",
    "REMOVE TABLE IF EXISTS nb_note;",
    "REMOVE ANALYZER IF EXISTS nb_english;",
    "REMOVE ANALYZER IF EXISTS nb_plain;",
]
client = await SurrealDBConnectionManager.get_client()
for statement in RESET:
    with contextlib.suppress(Exception):
        await client.query(statement, {})
major = int((await client.version()).rsplit("-", 1)[-1].split(".")[0])  # "surrealdb-3.3.4" -> 3
print(*Article.get_indexes(), sep="\n")

Index('nb_article_title', ('title',), search_analyzer=Analyzer('nb_english', ('class', 'blank'), ('lowercase', 'ascii', 'snowball(english)')), bm25=True, highlights=True)
Index('nb_article_body', ('body',), search_analyzer=Analyzer('nb_english', ('class', 'blank'), ('lowercase', 'ascii', 'snowball(english)')), bm25=True)
Index('nb_article_slug', ('slug',), unique=True)


## 4. Render the statements

`index_ddl()` renders the statements without a connection: the analyzers the indexes hold, each once, then one
`DEFINE INDEX` per index. It renders SurrealDB 3.x's `FULLTEXT` keyword, or 2.x's `SEARCH` with `line=2` — a script
for a 2.x server. The BM25 parameters are SurrealDB's defaults, `(1.2, 0.75)`; `bm25=(k1, b)` sets others.

In [3]:
print(*Article.index_ddl(), sep="\n")
print()
print(*Article.index_ddl(line=2), sep="\n")

DEFINE ANALYZER OVERWRITE `nb_english` TOKENIZERS class,blank FILTERS lowercase,ascii,snowball(english);
DEFINE INDEX OVERWRITE `nb_article_title` ON `nb_article` FIELDS `title` FULLTEXT ANALYZER `nb_english` BM25 HIGHLIGHTS;
DEFINE INDEX OVERWRITE `nb_article_body` ON `nb_article` FIELDS `body` FULLTEXT ANALYZER `nb_english` BM25;
DEFINE INDEX OVERWRITE `nb_article_slug` ON `nb_article` FIELDS `slug` UNIQUE;

DEFINE ANALYZER OVERWRITE `nb_english` TOKENIZERS class,blank FILTERS lowercase,ascii,snowball(english);
DEFINE INDEX OVERWRITE `nb_article_title` ON `nb_article` FIELDS `title` SEARCH ANALYZER `nb_english` BM25 HIGHLIGHTS;
DEFINE INDEX OVERWRITE `nb_article_body` ON `nb_article` FIELDS `body` SEARCH ANALYZER `nb_english` BM25;
DEFINE INDEX OVERWRITE `nb_article_slug` ON `nb_article` FIELDS `slug` UNIQUE;


## 5. Apply them

Call `define_table()` (v0.34.0) first: on a table that does not exist yet, SurrealDB's `DEFINE INDEX` would create it
`TYPE ANY SCHEMALESS PERMISSIONS NONE`. `define_indexes()` then reads what the database holds and the server line, and
applies what is missing or changed, one statement per request, with the line's keyword. It returns the statements
applied: on an unchanged database, `[]` — it is safe at start-up, and never rebuilds an unchanged index.

In [4]:
await Article.define_table()
applied = await Article.define_indexes()
print(*applied, sep="\n")
print()
print("the statements of index_ddl() for this line:", applied == Article.index_ddl(line=3 if major >= 3 else 2))
print("a second call:", await Article.define_indexes())

DEFINE ANALYZER OVERWRITE `nb_english` TOKENIZERS class,blank FILTERS lowercase,ascii,snowball(english);
DEFINE INDEX OVERWRITE `nb_article_title` ON `nb_article` FIELDS `title` FULLTEXT ANALYZER `nb_english` BM25 HIGHLIGHTS;
DEFINE INDEX OVERWRITE `nb_article_body` ON `nb_article` FIELDS `body` FULLTEXT ANALYZER `nb_english` BM25;
DEFINE INDEX OVERWRITE `nb_article_slug` ON `nb_article` FIELDS `slug` UNIQUE;

the statements of index_ddl() for this line: True
a second call: []


## 6. The corpus

Six articles, saved with `bulk_create()`: SurrealDB indexes each record as it is written.

In [5]:
corpus = [
    ("a1", "Quantum physics for beginners", "An introduction to quantum mechanics", "science"),
    ("a2", "Cooking with physics", "Heat and physics in the kitchen", "food"),
    ("a3", "Gardening in spring", "Plants and soil", "garden"),
    ("a4", "Quantum computing explained", "Qubits and gates", "science"),
    ("a5", "Baking bread at home", "Flour water salt", "food"),
    ("a6", "Sewing for beginners", "Needles and thread", "craft"),
]
created = await Article.objects().bulk_create(
    [Article(id=key, title=title, body=body, slug=key, topic=topic) for key, title, body, topic in corpus]
)
print("articles:", len(created))

articles: 6


## 7. Search

`search(title="physics")` keeps the records whose title matches: the analyzer stems and lower-cases both the text and
the search text. The text is bound, never written into the statement: it is data, whatever it holds. The terms of one
text are AND-ed (`"quantum physics"` needs both), and an empty text matches nothing.

In [6]:
print("physics:", sorted(article.title for article in await Article.objects().search(title="physics").exec()))
print("quantum physics:", [article.id for article in await Article.objects().search(title="quantum physics").exec()])
print("an empty text:", await Article.objects().search(title="").count())

physics: ['Cooking with physics', 'Quantum physics for beginners']
quantum physics: ['a1']
an empty text: 0


## 8. Score

Each field `search()` names gets a match reference, in call order: `0` here. `SearchScore(0)` in `annotate()` is that
match's BM25 score, a `float` set on each record as an attribute named after its alias, and `order_by("-relevance")`
sorts on it. "physics" is in two titles out of six: the index was built once, after §3 removed the table, so these
scores are the same on SurrealDB 3.x and 2.x.

In [7]:
from surreal_orm_lite import SearchScore

articles = await Article.objects().search(title="physics").annotate(relevance=SearchScore(0)).order_by("-relevance").exec()
for article in articles:
    print(article.id, article.title, round(article.relevance, 3))
print("never written back:", "relevance" in articles[0].model_dump())

a2 Cooking with physics 0.613
a1 Quantum physics for beginners 0.543
never written back: False


The scores depend on the line for a term found in more than half the documents, such as "and" in four bodies out of
six: the values below are SurrealDB 3.x's, all `0.0`, and a SurrealDB 2.x server prints values below zero. BM25 scores
depend on the line: SurrealDB 3.x gives a term found in more than half the documents no weight (a score of `0.0`) and
keeps its statistics in step with the records, while 2.x and the embedded engine score such a term below zero, count
an index's records again each time it is rebuilt, and keep counting deleted records. Compare scores within one query,
and upgrade to SurrealDB 3.x where scores must not depend on the index's history.

In [8]:
print(f"Server: SurrealDB {major}.x")
common = await Article.objects().search(body="and").annotate(score=SearchScore(0)).order_by("id").exec()
for article in common:
    print(article.id, article.body, round(article.score, 3))

Server: SurrealDB 3.x
a2 Heat and physics in the kitchen 0.0
a3 Plants and soil 0.0
a4 Qubits and gates 0.0
a6 Needles and thread 0.0


## 9. Highlight and offsets

`SearchHighlight(open_tag, close_tag, ref)` gives the field's text with each matched term between the tags — bound as
query variables too — and `SearchOffsets(ref)` where the terms sit: SurrealDB's `{"<item>": [{"s": <start>, "e":
<end>}]}`. Both need the index's `highlights=True`: on `body`, whose index has none, SurrealDB returns the text
unhighlighted and no offsets (`None`).

In [9]:
from surreal_orm_lite import SearchHighlight, SearchOffsets

highlighted = await (
    Article.objects()
    .search(title="quantum")
    .annotate(snippet=SearchHighlight("<b>", "</b>", 0), spans=SearchOffsets(0))
    .order_by("id")
    .exec()
)
for article in highlighted:
    print(article.id, article.snippet, article.spans)

plain = (
    await Article.objects()
    .search(body="kitchen")
    .annotate(snippet=SearchHighlight("<b>", "</b>", 0), spans=SearchOffsets(0))
    .get()
)
print(plain.id, plain.snippet, plain.spans)

a1 <b>Quantum</b> physics for beginners {'0': [{'e': 7, 's': 0}]}
a4 <b>Quantum</b> computing explained {'0': [{'e': 7, 's': 0}]}
a2 Heat and physics in the kitchen None


## 10. With the rest of the queryset

`search()` applies wherever `filter()` does, AND-ed with it: two fields (references `0` and `1`), `filter()`,
`count()`, `values()` groups with `Count()`, and a `Subquery`, whose references start again at `0`.

In [10]:
from surreal_orm_lite import Count, Subquery

both = await Article.objects().search(title="quantum", body="qubits").annotate(t=SearchScore(0), b=SearchScore(1)).exec()
print("two fields:", [(article.id, article.t > 0, article.b > 0) for article in both])
science = await Article.objects().filter(topic="science").search(title="quantum").order_by("id").exec()
print("with filter():", [article.id for article in science])
print("count():", await Article.objects().search(title="quantum").count())
groups = await Article.objects().search(title="beginners").values("topic").annotate(n=Count()).exec()
print("values() + Count():", sorted(groups, key=lambda group: group["topic"]))
beginners = Subquery(Article.objects().search(title="beginners"))
print("a Subquery:", [article.id for article in await Article.objects().filter(id__in=beginners).search(body="thread").exec()])

two fields: [('a4', True, True)]
with filter(): ['a1', 'a4']
count(): 2
values() + Count(): [{'n': 1, 'topic': 'craft'}, {'n': 1, 'topic': 'science'}]
a Subquery: ['a6']


## 11. What is refused

Each raises `SurrealDbError`:

- a search on a field without a full-text index — `slug`'s is `UNIQUE` — names the field; the server's own error, which
  on SurrealDB 3.x quotes the search text, is never shown;
- `as_of()` (v0.26.0), before any request: the index holds the records' current text, so a time-travel read would
  select records by their text now;
- `live()` and `watch()`, before any request: SurrealDB accepts a live query whose condition holds a full-text match,
  but never notifies it. Search in a subquery instead.

In [11]:
from datetime import timedelta

from surreal_orm_lite import SurrealDbError

try:
    await Article.objects().search(slug="a1").exec()
except SurrealDbError as error:
    print(error)
print()
try:
    Article.objects().search(title="physics").as_of(timedelta(days=1))
except SurrealDbError as error:
    print(error)
print()
try:
    Article.objects().search(title="physics").live()
except SurrealDbError as error:
    print(error)

search(): no full-text index on field 'slug'. SurrealDB matches only a field with a FULLTEXT (3.x) or SEARCH (2.x) index: declare Index(..., search_analyzer=...) on the model and apply define_indexes().

search() cannot be combined with as_of(): a full-text match reads the index, which holds the records' current text, so a time-travel read would select records by their text now. Run the search without as_of().

A live query cannot honour search(): SurrealDB accepts a LIVE SELECT whose condition holds a full-text match but never notifies it. Search in a subquery instead — filter(id__in=Subquery(Model.objects().search(...))) — which notifies the records created or updated to match, not their deletion.


A live query whose subquery searches — `filter(id__in=Subquery(Article.objects().search(title="physics")))` — notifies
the records created or updated to match, not a deletion, nor an update that makes a record stop matching. Before a live query whose subquery searches, the ORM checks that each
searched field has a full-text index and raises `SurrealDbError` naming the field otherwise, before the live query
starts: without the index, SurrealDB 3.x would accept the live query and never notify. Do not remove that index while
the live query runs.

In [12]:
unindexed = Subquery(Article.objects().search(slug="a1"))
try:
    async with Article.objects().filter(id__in=unindexed).live() as stream:
        print("live query started")  # not reached
except SurrealDbError as error:
    print(error)

search(): no full-text index on field 'slug'. SurrealDB matches only a field with a FULLTEXT (3.x) or SEARCH (2.x) index: declare Index(..., search_analyzer=...) on the model and apply define_indexes().


## 12. Changing an analyzer

SurrealDB keeps the tokens of the records an index holds when their analyzer changes: a search tokenised the new way
would miss them until the index is rebuilt. So when `define_indexes()` redefines an analyzer, it rebuilds every index of
the database that uses it (`REBUILD INDEX`), and logs one warning naming them (logger `surreal_orm_lite.search`).

`Note`'s analyzer `nb_plain` keeps the case (`ascii` only): "physics" does not match "Physics Rocks". `NoteV2`, the same
table in a later release, adds `lowercase`: `define_indexes()` redefines the analyzer and rebuilds the index, and the
record now matches. After a rebuild, compare matches, not scores: 2.x counts the index's records again.

In [13]:
class Note(BaseSurrealModel):
    model_config = SurrealConfigDict(
        table_name="nb_note",
        indexes=[Index("nb_note_text", "text", search_analyzer=Analyzer("nb_plain", ["blank"], ["ascii"]))],
    )

    id: str | None = None
    text: str


await Note.define_table()
print(*await Note.define_indexes(), sep="\n")
await Note(id="n1", text="Physics Rocks").save()
print("physics:", [note.id for note in await Note.objects().search(text="physics").exec()])

DEFINE ANALYZER OVERWRITE `nb_plain` TOKENIZERS blank FILTERS ascii;
DEFINE INDEX OVERWRITE `nb_note_text` ON `nb_note` FIELDS `text` FULLTEXT ANALYZER `nb_plain`;
physics: []


In [14]:
class NoteV2(Note):
    model_config = SurrealConfigDict(
        table_name="nb_note",
        indexes=[Index("nb_note_text", "text", search_analyzer=Analyzer("nb_plain", ["blank"], ["lowercase", "ascii"]))],
    )


print(*await NoteV2.define_indexes(), sep="\n")
print("physics:", [note.id for note in await NoteV2.objects().search(text="physics").exec()])

Analyzer 'nb_plain' redefined; 1 index(es) rebuilt: nb_note.nb_note_text


DEFINE ANALYZER OVERWRITE `nb_plain` TOKENIZERS blank FILTERS lowercase,ascii;
REBUILD INDEX `nb_note_text` ON `nb_note`;
physics: ['n1']


## 13. The line

`define_indexes()` read the line once, with `client.version()`, and wrote its keyword: `FULLTEXT` here, `SEARCH` on a
2.x server. `INFO FOR TABLE` reads the index back as SurrealDB stores it, with the BM25 defaults (2.x adds its other
defaults too).

In [15]:
version = await client.version()
print(f"Server: SurrealDB {int(version.rsplit('-', 1)[-1].split('.')[0])}.x")
print("keyword the ORM wrote:", "FULLTEXT" if any(" FULLTEXT " in statement for statement in applied) else "SEARCH")
print("stored:", (await client.query("INFO FOR TABLE nb_article;", {}))["indexes"]["nb_article_title"])

Server: SurrealDB 3.x
keyword the ORM wrote: FULLTEXT
stored: DEFINE INDEX nb_article_title ON nb_article FIELDS title FULLTEXT ANALYZER nb_english BM25(1.2,0.75) HIGHLIGHTS


## 14. Cleanup

`remove_index()` removes an index by name; `define_indexes()` never removes one. `remove_analyzer()` refuses an analyzer
an index uses, on every line, with SurrealDB 3.x's reason — on SurrealDB 2.x the server would remove it and then
refuse every write to the table. Then the tables and the analyzers go, and the connection is closed.

In [16]:
print(await Article.remove_index("nb_article_body"))
try:
    await SurrealDBConnectionManager.remove_analyzer("nb_english")
except SurrealDbError as error:
    print(error)
for statement in RESET[:2]:
    await client.query(statement, {})
print(await SurrealDBConnectionManager.remove_analyzer("nb_english"))
print(await SurrealDBConnectionManager.remove_analyzer("nb_plain"))
await SurrealDBConnectionManager.close_connection()
print("Done")

REMOVE INDEX IF EXISTS `nb_article_body` ON `nb_article`;
Can't apply analyzer removal: 'nb_english' -> The analyzer 'nb_english' is in use by index 'nb_article_title' on table 'nb_article' and cannot be removed
REMOVE ANALYZER IF EXISTS `nb_english`;
REMOVE ANALYZER IF EXISTS `nb_plain`;
Done
